# 05 - Moteur de Decision Clinique (SAD)

Ce notebook couvre le livrable 6.1 en une seule execution:
1. Introduction au SAD (classification vs decision).
2. Utilisation de sorties de modeles calibres (RegLog, MLP, CNN).
3. Application du moteur de decision clinique.
4. Simulation de 20 rapports patients.
5. Analyse critique et ethique.

## 1) Introduction - Classification vs Decision

- **Classification**: le modele predit une classe et des probabilites.
- **Decision clinique**: le SAD transforme ces probabilites en action, priorite et niveau de revision humaine.
- **Objectif**: reduire le risque (surtout faux negatifs) et structurer le triage medical.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.decision.engine import generer_recommandation
from src.decision.rules import appliquer_regle_securite_negatif
from src.decision.triage import appliquer_triage
from src.config.thresholds import CostParameters

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
RNG = np.random.default_rng(42)

print("Imports OK")

Imports OK


## 2) Modeles calibres (RegLog, MLP, CNN)

Dans ce notebook, on simule des sorties de probabilites **deja calibrees** venant des notebooks precedents:
- `02_reglog_calibration.ipynb`
- `03_mlp_incertitude.ipynb`
- `04_cnn_temperature_scaling.ipynb`

Le but ici est de valider le pipeline decisionnel SAD sans relancer des entrainements longs.

In [3]:
def simulate_calibrated_probs(model_name: str, n: int, rng: np.random.Generator) -> np.ndarray:
    # Dirichlet -> vecteurs de proba valides, puis leger boost selon le modele
    base = rng.dirichlet(alpha=[1.2, 1.2, 1.4, 1.0], size=n)

    if model_name == "reglog":
        base[:, 2] += 0.05
    elif model_name == "mlp":
        base[:, 0] += 0.04
    elif model_name == "cnn":
        base[:, 0] += 0.07
        base[:, 1] += 0.03

    base = np.clip(base, 1e-8, None)
    base = base / base.sum(axis=1, keepdims=True)
    return base

n_cases = 20
p_reglog = simulate_calibrated_probs("reglog", n_cases, RNG)
p_mlp = simulate_calibrated_probs("mlp", n_cases, RNG)
p_cnn = simulate_calibrated_probs("cnn", n_cases, RNG)

# Fusion simple: on privilegie legerement le CNN (plus robuste en vision).
p_fused = 0.2 * p_reglog + 0.3 * p_mlp + 0.5 * p_cnn
p_fused = p_fused / p_fused.sum(axis=1, keepdims=True)

max_probs = p_fused.max(axis=1)
print(f"Cas incertains (max_prob < 0.7): {(max_probs < 0.7).sum()} / {n_cases}")

Cas incertains (max_prob < 0.7): 20 / 20


## 3) Moteur de decision - Regles metier

Regles appliquees:
- `>= 0.85` : Diagnostic automatique valide.
- `>= 0.65` : Diagnostic probable, revision junior.
- `>= 0.50` : Cas incertain, revision senior.
- `< 0.50` : Incertitude elevee, double lecture + IRM complementaire.

Securite faux negatifs:
- Si classe predite `notumor` et confiance `< 0.95`, verification obligatoire.

In [4]:
def run_decision_pipeline(prob_vector: np.ndarray, patient_id: str):
    decision = generer_recommandation(prob_vector, CLASSES, patient_id=patient_id)
    decision = appliquer_regle_securite_negatif(decision)
    decision = appliquer_triage(decision)
    return decision

decisions = []
for i in range(n_cases):
    pid = f"P_{i+1:05d}"
    d = run_decision_pipeline(p_fused[i], pid)
    decisions.append(d)

df_decisions = pd.DataFrame([
    {
        "patient_id": d.patient_id,
        "classe_predite": d.classe_predite,
        "confiance": d.confiance,
        "niveau_confiance": d.niveau_confiance,
        "decision": d.decision,
        "action": d.action_recommandee,
        "priorite": d.priorite,
        "revision_requise": d.revision_requise,
        "alerte_securite": d.alerte_securite,
    }
    for d in decisions
])

df_decisions.head()

,patient_id,classe_predite,confiance,niveau_confiance,decision,action,priorite,revision_requise,alerte_securite
0,P_00001,meningioma,0.425041,TRES_FAIBLE,Incertitude elevee,Double lecture obligatoire + IRM complementaire,Elevée (24h),True,False
1,P_00002,meningioma,0.310241,TRES_FAIBLE,Incertitude elevee,Double lecture obligatoire + IRM complementaire,Elevée (24h),True,False
2,P_00003,glioma,0.349420,TRES_FAIBLE,Incertitude elevee,Double lecture obligatoire + IRM complementaire,URGENTE (12h),True,False
3,P_00004,glioma,0.375201,TRES_FAIBLE,Incertitude elevee,Double lecture obligatoire + IRM complementaire,URGENTE (12h),True,False
4,P_00005,notumor,0.328858,TRES_FAIBLE,Prédiction 'Pas de tumeur' à 32.9% confiance -...,Verification obligatoire (risque faux negatif)...,URGENTE (12h),True,True


## 4) Simulation - 20 rapports patient

Generation d un rapport textuel formate pour chaque patient.

In [5]:
from datetime import datetime

def creer_rapport_decision(patient_id: str, decision_obj) -> str:
    scores = sorted(decision_obj.probabilites.items(), key=lambda x: x[1], reverse=True)
    certitude_ok = "OK" if decision_obj.niveau_confiance == "HAUTE" else "A_VERIFIER"

    lines = [
        "========================================",
        "RAPPORT D AIDE A LA DECISION",
        "========================================",
        f"Patient ID: {patient_id} Date: {datetime.now().strftime('%d/%m/%Y')}",
        "",
        "PREDICTION PRINCIPALE",
        "---",
        f"Classe: {decision_obj.classe_predite}",
        f"Confiance: {decision_obj.confiance:.1%}",
        f"Niveau de certitude: {decision_obj.niveau_confiance} [{certitude_ok}]",
        "",
        "SCORES PAR CLASSE",
        "---",
    ]

    for classe, p in scores:
        lines.append(f"- {classe}: {p:.1%}")

    lines.extend([
        "",
        "RECOMMANDATIONS CLINIQUES",
        "---",
        f"Diagnostic: {decision_obj.decision}",
        f"Action: {decision_obj.action_recommandee}",
        f"Priorite: {decision_obj.priorite}",
        f"Revision humaine: {'Obligatoire' if decision_obj.revision_requise else 'Optionnelle'}",
        "",
        "ELEMENTS D ATTENTION",
        "---",
    ])

    if decision_obj.alerte_securite:
        lines.append("- Verification obligatoire (risque faux negatif)")
    if decision_obj.classe_predite == "glioma":
        lines.append("- Tumeur maligne suspectee")
        lines.append("- Orientation oncologie recommandee")
    if decision_obj.niveau_confiance in {"FAIBLE", "TRES_FAIBLE"}:
        lines.append("- Relecture humaine prioritaire")

    lines.append("========================================")
    return "\n".join(lines)

rapports = [creer_rapport_decision(d.patient_id, d) for d in decisions]
print(f"Nombre de rapports generes: {len(rapports)}")

# Afficher 3 exemples (les 20 sont dans la variable 'rapports').
for i in range(3):
    print("\n" + rapports[i] + "\n")

Nombre de rapports generes: 20

RAPPORT D AIDE A LA DECISION
Patient ID: P_00001 Date: 05/03/2026

PREDICTION PRINCIPALE
---
Classe: meningioma
Confiance: 42.5%
Niveau de certitude: TRES_FAIBLE [A_VERIFIER]

SCORES PAR CLASSE
---
- meningioma: 42.5%
- notumor: 24.3%
- pituitary: 17.8%
- glioma: 15.4%

RECOMMANDATIONS CLINIQUES
---
Diagnostic: Incertitude elevee
Action: Double lecture obligatoire + IRM complementaire
Priorite: Elevée (24h)
Revision humaine: Obligatoire

ELEMENTS D ATTENTION
---
- Relecture humaine prioritaire


RAPPORT D AIDE A LA DECISION
Patient ID: P_00002 Date: 05/03/2026

PREDICTION PRINCIPALE
---
Classe: meningioma
Confiance: 31.0%
Niveau de certitude: TRES_FAIBLE [A_VERIFIER]

SCORES PAR CLASSE
---
- meningioma: 31.0%
- notumor: 29.9%
- glioma: 23.2%
- pituitary: 15.8%

RECOMMANDATIONS CLINIQUES
---
Diagnostic: Incertitude elevee
Action: Double lecture obligatoire + IRM complementaire
Priorite: Elevée (24h)
Revision humaine: Obligatoire

ELEMENTS D ATTENTION
---


In [6]:
# Resume des 20 cas
display_cols = [
    "patient_id",
    "classe_predite",
    "confiance",
    "niveau_confiance",
    "priorite",
    "revision_requise",
    "alerte_securite",
]

df_decisions[display_cols].sort_values(["priorite", "confiance"], ascending=[True, False])

,patient_id,classe_predite,confiance,niveau_confiance,priorite,revision_requise,alerte_securite
14,P_00015,meningioma,0.498290,TRES_FAIBLE,Elevée (24h),True,False
16,P_00017,meningioma,0.453157,TRES_FAIBLE,Elevée (24h),True,False
11,P_00012,meningioma,0.427585,TRES_FAIBLE,Elevée (24h),True,False
0,P_00001,meningioma,0.425041,TRES_FAIBLE,Elevée (24h),True,False
18,P_00019,meningioma,0.412661,TRES_FAIBLE,Elevée (24h),True,False
6,P_00007,meningioma,0.389443,TRES_FAIBLE,Elevée (24h),True,False
12,P_00013,pituitary,0.387850,TRES_FAIBLE,Elevée (24h),True,False
8,P_00009,meningioma,0.363547,TRES_FAIBLE,Elevée (24h),True,False
10,P_00011,pituitary,0.354340,TRES_FAIBLE,Elevée (24h),True,False
13,P_00014,pituitary,0.319684,TRES_FAIBLE,Elevée (24h),True,False


## 5) Analyse metier + critique et ethique

In [7]:
# Metriques orientees metier
# Simulation de labels reels pour evaluer le cout (demo).
y_true_idx = RNG.integers(low=0, high=len(CLASSES), size=n_cases)
y_pred_idx = np.array([CLASSES.index(d.classe_predite) for d in decisions])

is_notumor_true = y_true_idx == CLASSES.index("notumor")
is_notumor_pred = y_pred_idx == CLASSES.index("notumor")

fn = int(((~is_notumor_pred) & is_notumor_true).sum())
fp = int((is_notumor_pred & (~is_notumor_true)).sum())
revisions = int(df_decisions["revision_requise"].sum())

costs = CostParameters()
cost_total = fn * costs.faux_negatif + fp * costs.faux_positif + revisions * costs.revision

auto_coverage = float((~df_decisions["revision_requise"]).mean())
high_conf_mask = df_decisions["confiance"] > 0.85
if high_conf_mask.any():
    high_conf_acc = float((y_true_idx[high_conf_mask.values] == y_pred_idx[high_conf_mask.values]).mean())
else:
    high_conf_acc = np.nan

metrics = {
    "taux_couverture_automatique": auto_coverage,
    "accuracy_confiance_gt_085": high_conf_acc,
    "FN": fn,
    "FP": fp,
    "revision": revisions,
    "cost_total": cost_total,
}

pd.Series(metrics)

taux_couverture_automatique       0.0
accuracy_confiance_gt_085         NaN
FN                                5.0
FP                                1.0
revision                         20.0
cost_total                     6100.0
dtype: float64